# Steer Prompt

Visualizes the effect of prepending ideological system-prompt prefixes on LLM opinion shifts (Appendix E): per-level Bayesian intercepts, panel grids, and slope heatmaps. Reads from `outputs/steer_prompt_bayesian/`.

In [1]:
import os
os.chdir('../')

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import glob
import json
from src import utils
from IPython.display import clear_output
import numpy as np
from scipy import stats

os.environ['PATH'] = f"{os.path.expanduser('~/.TinyTeX/bin/x86_64-linux')}:{os.environ['PATH']}"

sns.set_theme(context='paper', style='ticks', font_scale=1)

In [ ]:
name = "steer_prompt"
width_pt = 469
palette = sns.color_palette('husl', 5)
steer_palette = sns.diverging_palette(240, 10, n=7)

LEVEL_LABELS_MAP = {
    "political": [
        'Strongly\nconservative', 'Conservative', 'Leaning\nconservative',
        'Neutral',
        'Leaning\nliberal', 'Liberal', 'Strongly\nliberal'
    ],
}

## Configuration

In [ ]:
# model_name = "mistralai/Ministral-3-8B-Instruct-2512"
# model_name = "meta-llama/Llama-3.1-8B-Instruct"
model_name = "google/gemma-3-12b-it"
# model_name = "Qwen/Qwen3-8B"

dataset = "ukp"
# dataset = "semeval"

task = "writing"
# task = "improvement"
# task = "legacy_semeval"

quantification_method = "centroid"

prompt_variant = "political"
# prompt_variant = "legacy_political"
# prompt_variant = "legacy_subtle"
# prompt_variant = "legacy_identity"

# UKP topics
topic = "abortion"
# topic = "cloning"
# topic = "death_penalty"
# topic = "gun_control"
# topic = "marijuana_legalization"
# topic = "minimum_wage"
# topic = "nuclear_energy"
# topic = "school_uniforms"

# SemEval topics
# topic = "atheism"
# topic = "acknowledging_climate_change"
# topic = "feminism"
# topic = "hillary_clinton"
# topic = "abortion"
# topic = "donald_trump"

LEVEL_LABELS = LEVEL_LABELS_MAP[prompt_variant]

## Bias by topic and steering level (wide)

In [ ]:
import matplotlib.lines as mlines

if dataset == "semeval":
    topics = ["atheism", "acknowledging_climate_change", "feminism", "hillary_clinton", "abortion", "donald_trump"]
else:
    topics = ["abortion", "cloning", "death_penalty", "gun_control",
              "marijuana_legalization", "minimum_wage", "nuclear_energy", "school_uniforms"]

model_sanitized = model_name.replace('/', '_')

# Load per-(topic, level) Bayesian intercepts
records = []
for t_i, t in enumerate(topics):
    for level in range(1, 8):
        jf = f"outputs/steer_prompt_bayesian/{dataset}__{task}__{model_sanitized}__{t}__{quantification_method}__prompt_variant={prompt_variant}__level={level}.json"
        if not os.path.exists(jf):
            continue
        with open(jf, 'r') as f:
            d = json.load(f)
        ic = d['model_direction']['intercept']
        records.append({
            'topic': t, 'topic_idx': t_i, 'level': level,
            'mean': ic['mean'], 'lower_95': ic['lower_95'], 'upper_95': ic['upper_95'],
        })

bias_df = pd.DataFrame(records)

# Layout: topics at integer x positions; 7 levels spread within each topic
n_topics = len(topics)
within_offsets = np.linspace(-0.35, 0.35, 7)
bias_df['x'] = bias_df['topic_idx'] + bias_df['level'].map(lambda lv: within_offsets[int(lv) - 1])

# Color encoding by steering level
# Level 1 (Strong conservative) -> cmap(1.0) = red end.
# Level 4 (Neutral)             -> cmap(0.5) = silver/white midpoint.
# Level 7 (Strong liberal)      -> cmap(0.0) = blue end.
ideology_cmap = sns.color_palette("vlag", as_cmap=True)

def _level_to_color(level):
    return ideology_cmap(1.0 - (int(level) - 1) / 6.0)

utils.latexify()
fig_width, fig_height = utils.get_fig_dim(width_pt, fraction=0.6)
fig, ax = plt.subplots(figsize=(fig_width * 3, fig_height))

# Connecting line per topic (drawn under the points)
for t_i in range(n_topics):
    sub = bias_df[bias_df['topic_idx'] == t_i].sort_values('level')
    if len(sub) < 2:
        continue
    ax.plot(sub['x'], sub['mean'], color='gray', linewidth=0.8, zorder=1)

# Errorbar points
for _, row in bias_df.iterrows():
    color = _level_to_color(row['level'])
    ax.errorbar(
        x=row['x'], y=row['mean'],
        yerr=[[row['mean'] - row['lower_95']], [row['upper_95'] - row['mean']]],
        fmt='o', capsize=2, color=color,
        elinewidth=1.2, markersize=5, capthick=1.2, zorder=2,
    )

ax.axhline(y=0, color='gray', linestyle='--', linewidth=1, zorder=0)

# Topic labels at integer x positions
def _topic_label(t):
    txt = t.replace('_', ' ')
    txt = txt.title() if t in ('donald_trump', 'hillary_clinton') else txt[0].upper() + txt[1:]
    if txt == "Acknowledging climate change":
        return "Acknowledging\nclimate change"
    return txt.replace(' ', '\n')

ax.set_xticks(range(n_topics))
ax.set_xticklabels([_topic_label(t) for t in topics], fontsize=8)
for label in ax.get_xticklabels():
    label.set_multialignment('center')

ax.set_xlim([-0.6, n_topics - 0.4])
ax.set_xlabel("Topic")
ax.set_ylabel(r"Bias towards ``in favor''")

# Legend: one entry per ideology level, in the conservative -> liberal order
legend_handles = [
    mlines.Line2D([], [], color=_level_to_color(lv), marker='o', linestyle='None',
                  markersize=6, label=LEVEL_LABELS_MAP[prompt_variant][lv - 1].replace('\n', ' '))
    for lv in range(1, 8)
]
ax.legend(handles=legend_handles, loc='lower center', bbox_to_anchor=(0.5, 1.02),
          ncol=7, frameon=False, fontsize=8, handletextpad=0.4, columnspacing=1.0)

sns.despine(ax=ax)
fig.tight_layout()
fig.savefig(f'figures/{name}__bias_by_topic_and_level_ideology__{dataset}__{model_sanitized}__{prompt_variant}.pdf',
            dpi=300, bbox_inches='tight')
plt.close()